In [1]:
import pandas as pd
from datetime import datetime
import numpy as np
import matplotlib.pyplot as plt
import os

In [2]:
datasets = [
    "data/aligned/XLB_aligned.txt","data/aligned/XLE_aligned.txt","data/aligned/XLF_aligned.txt",
    "data/aligned/XLI_aligned.txt","data/aligned/XLK_aligned.txt","data/aligned/XLP_aligned.txt",
    "data/aligned/XLU_aligned.txt","data/aligned/XLY_aligned.txt"
]

dataframes = []

for filename in datasets:
    df = pd.read_csv(
        filename,
        sep='\t',
        index_col=0,
        parse_dates=True
    )
    df.columns = ["Close"]
    dataframes.append(df)
    print(df.head())
    print(df.index.dtype)



                       Close
2009-01-02 09:29:00  16.1147
2009-01-02 09:30:00  16.1927
2009-01-02 09:31:00  16.1785
2009-01-02 09:32:00  16.2210
2009-01-02 09:33:00  16.2139
datetime64[ns]
                      Close
2009-01-02 09:29:00  28.737
2009-01-02 09:30:00  28.875
2009-01-02 09:31:00  28.845
2009-01-02 09:32:00  28.863
2009-01-02 09:33:00  28.821
datetime64[ns]
                      Close
2009-01-02 09:29:00  7.6109
2009-01-02 09:30:00  7.5866
2009-01-02 09:31:00  7.5866
2009-01-02 09:32:00  7.5806
2009-01-02 09:33:00  7.5624
datetime64[ns]
                       Close
2009-01-02 09:29:00  17.1388
2009-01-02 09:30:00  17.1461
2009-01-02 09:31:00  17.1461
2009-01-02 09:32:00  17.1461
2009-01-02 09:33:00  17.1095
datetime64[ns]
                       Close
2009-01-02 09:30:00  12.3457
2009-01-02 09:31:00  12.3217
2009-01-02 09:32:00  12.3138
2009-01-02 09:33:00  12.3058
2009-01-02 09:34:00  12.3298
datetime64[ns]
                       Close
2009-01-02 09:29:00  15.4782
2009-01-0

In [3]:
dataframes[0]

,Close
2009-01-02 09:29:00,16.1147
2009-01-02 09:30:00,16.1927
2009-01-02 09:31:00,16.1785
2009-01-02 09:32:00,16.2210
2009-01-02 09:33:00,16.2139
...,...
2024-12-30 15:57:00,83.8450
2024-12-30 15:58:00,83.8300
2024-12-30 15:59:00,83.8000
2024-12-30 16:00:00,83.8000


In [4]:
df_merged = pd.concat(dataframes, axis=1)
df_merged.columns = ["XLB","XLE","XLF","XLI","XLK","XLP","XLU","XLY"]

In [5]:
df_merged

,XLB,XLE,XLF,XLI,XLK,XLP,XLU,XLY
2009-01-02 09:29:00,16.1147,28.7370,7.6109,17.1388,NaN,15.4782,16.5942,17.6183
2009-01-02 09:30:00,16.1927,28.8750,7.5866,17.1461,12.3457,15.4846,16.5149,17.6755
2009-01-02 09:31:00,16.1785,28.8450,7.5866,17.1461,12.3217,15.4846,16.4979,17.6101
2009-01-02 09:32:00,16.2210,28.8630,7.5806,17.1461,12.3138,15.4911,16.4809,17.6020
2009-01-02 09:33:00,16.2139,28.8210,7.5624,17.1095,12.3058,15.4782,16.4752,17.5611
...,...,...,...,...,...,...,...,...
2024-12-30 15:57:00,83.8450,84.6333,48.3250,132.0300,234.9100,78.4300,75.8150,226.3601
2024-12-30 15:58:00,83.8300,84.6000,48.3150,132.0000,234.7300,78.4050,75.7900,226.2400
2024-12-30 15:59:00,83.8000,84.5300,48.2650,131.9100,234.4500,78.3700,75.7650,226.0300
2024-12-30 16:00:00,83.8000,84.5500,48.2800,131.9200,234.4600,78.3700,75.7600,226.0500


In [6]:
def get_realized_cov_sub(day_data, date, prev_close=None):
    first_ts = pd.Timestamp(f"{date} 09:29")
    if first_ts in day_data.index:
        first_price = day_data.loc[first_ts]
    elif prev_close is not None:
        first_price = prev_close
    else:
        first_price = day_data.iloc[0]

    first_time = day_data.index[0] - pd.Timedelta(minutes=1)
    df_for_return = pd.concat([
        pd.DataFrame([first_price], index=[first_time]),
        day_data
    ])

    returns = np.log(df_for_return / df_for_return.shift(1)).dropna()
    
    return returns

In [7]:

    
prev_close = None
returns = []

for date, day_data in df_merged.groupby(df_merged.index.date):
    rets = get_realized_cov_sub(day_data, date, prev_close)

    rets = rets[rets.index.time != pd.to_datetime("20:00").time()]
    
    returns.append(rets)
    prev_close = day_data.iloc[-1]

In [8]:
returns[600].tail(11)

,XLB,XLE,XLF,XLI,XLK,XLP,XLU,XLY
2011-05-25 15:50:00,0.000274,0.000127,0.000636,0.000141,0.000000,0.000363,0.000048,0.000388
2011-05-25 15:51:00,0.000239,0.000000,0.000000,-0.000141,0.000000,-0.000227,-0.000381,-0.000119
2011-05-25 15:52:00,-0.000650,-0.000529,0.000000,-0.000528,-0.000422,0.000454,-0.000143,-0.000030
2011-05-25 15:53:00,-0.000376,-0.000254,0.000000,0.000000,0.000000,-0.000591,-0.000143,-0.000119
2011-05-25 15:54:00,-0.001027,-0.000677,-0.000636,-0.000528,0.000000,0.000000,-0.000572,-0.000626
2011-05-25 15:55:00,0.000000,-0.000127,0.000000,-0.000282,-0.000188,-0.000318,-0.000334,-0.000239
2011-05-25 15:56:00,-0.000240,-0.000381,-0.000328,-0.000282,-0.000188,-0.000318,-0.000286,-0.000119
2011-05-25 15:57:00,0.000754,0.000254,0.000328,0.000528,0.000188,0.000000,0.000000,0.000119
2011-05-25 15:58:00,0.000274,0.000254,0.000000,0.000246,0.000188,0.000318,0.000620,0.000507
2011-05-25 15:59:00,-0.000274,-0.000254,-0.000975,-0.000246,-0.000375,-0.000182,0.000143,-0.000268


In [9]:
def get_sub_rc(daily_returns):
    R = daily_returns.values
    subsamples = [R[k::5] for k in range(5)]
    rcovs = [r.T @ r for r in subsamples if len(r) > 0]
    rcov = sum(rcovs) / len(rcovs)
    return rcov

In [10]:
rcs = []
for i in returns:
    rc = get_sub_rc(i)
    indices = np.tril_indices_from(rc)
    lower_tri_1d = rc[indices]
    rcs.append(lower_tri_1d)

df_rcs = pd.DataFrame(rcs)

In [11]:
df_rcs

,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,35
0,0.000052,0.000037,0.000066,0.000046,0.000056,0.000123,0.000033,0.000035,0.000049,0.000042,...,0.000014,0.000030,3.437108e-05,3.830956e-05,0.000048,0.000035,0.000028,0.000018,2.101923e-05,0.000056
1,0.000120,0.000091,0.000120,0.000093,0.000091,0.000147,0.000067,0.000066,0.000072,0.000077,...,0.000030,0.000053,7.076944e-05,6.676677e-05,0.000078,0.000058,0.000048,0.000035,4.154894e-05,0.000075
2,0.000095,0.000069,0.000113,0.000069,0.000072,0.000127,0.000056,0.000058,0.000064,0.000065,...,0.000029,0.000049,5.539065e-05,6.049568e-05,0.000066,0.000052,0.000050,0.000028,4.114226e-05,0.000066
3,0.000096,0.000075,0.000147,0.000063,0.000077,0.000118,0.000049,0.000058,0.000059,0.000060,...,0.000014,0.000037,4.081903e-05,5.068724e-05,0.000054,0.000039,0.000041,0.000020,2.207400e-05,0.000054
4,0.000106,0.000083,0.000124,0.000075,0.000083,0.000126,0.000057,0.000065,0.000063,0.000070,...,0.000019,0.000037,5.607099e-05,6.419590e-05,0.000059,0.000054,0.000045,0.000031,2.931818e-05,0.000068
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3987,0.000015,0.000012,0.000024,0.000015,0.000017,0.000025,0.000014,0.000014,0.000019,0.000020,...,0.000011,0.000023,1.272642e-05,1.271461e-05,0.000019,0.000018,0.000026,0.000008,1.196133e-05,0.000038
3988,0.000011,0.000008,0.000017,0.000008,0.000009,0.000014,0.000008,0.000009,0.000010,0.000011,...,0.000007,0.000014,6.412485e-06,4.281973e-06,0.000008,0.000008,0.000015,0.000002,1.688689e-06,0.000021
3989,0.000006,0.000004,0.000010,0.000002,0.000003,0.000006,0.000002,0.000003,0.000004,0.000005,...,0.000002,0.000008,7.090624e-07,6.197380e-07,0.000004,0.000004,0.000007,0.000002,8.450104e-07,0.000011
3990,0.000011,0.000010,0.000016,0.000010,0.000011,0.000014,0.000010,0.000010,0.000011,0.000012,...,0.000006,0.000011,9.947298e-06,9.934542e-06,0.000013,0.000012,0.000018,0.000007,7.463896e-06,0.000021


In [12]:
#was incorrect this should be fine
def get_sub_rc(day_prices):
    P = day_prices.values 
    K = 5
    rcovs = []
    for k in range(K):
        sub_prices = P[k::K]
        if len(sub_prices) < 2:
            continue
        sub_returns = np.diff(np.log(sub_prices), axis=0)
        rcovs.append(sub_returns.T @ sub_returns)
    return sum(rcovs) / len(rcovs)

rcs = []
daily_closes = []
for date, day_data in df_merged.groupby(df_merged.index.date):
    day_prices = day_data.between_time("09:30", "16:00")
    rc = get_sub_rc(day_prices)
    indices = np.tril_indices_from(rc)
    rcs.append(rc[indices])

    daily_closes.append(day_prices.iloc[-1])

df_rcs = pd.DataFrame(rcs)
df_closes = pd.DataFrame(daily_closes)
df_returns = np.log(df_closes / df_closes.shift(1)).dropna()

# save
df_returns.to_csv("data/daily_returns.txt", sep='\t')
print(f"RC shape: {df_rcs.shape}")
print(f"Returns shape: {df_returns.shape}")

RC shape: (3992, 36)
Returns shape: (3991, 8)


In [13]:
output_dir = "data/"
os.makedirs(output_dir, exist_ok=True)
filename = "rc_subsampled.txt"
path = os.path.join(output_dir, filename)
    
df_rcs.to_csv(path, sep='\t', index=False, header=False) 
print(f"Saved rcovs to {path}")

Saved rcovs to data/rc_subsampled.txt


In [14]:
def parzen_kernel(x):
    x = abs(x)
    if x <= 0.5:
        return 1 - 6*x**2 + 6*x**3
    elif x <= 1:
        return 2*(1 - x)**3
    else:
        return 0

In [15]:
def compute_kernel_rcov(R, H):
    n, d = R.shape
    Sigma = np.zeros((d, d))
    Sigma += R.T @ R
    for h in range(1, H + 1):
        weight = parzen_kernel(h / H)
        Gamma_h = R[h:].T @ R[:-h]
        Sigma += weight * (Gamma_h + Gamma_h.T)
    
    return Sigma

In [16]:
rcs = []
dates = []
for date, day_data in df_merged.groupby(df_merged.index.date):
    day_prices = day_data.between_time("09:30", "16:00")
    if len(day_prices) < 2:
        continue
    P = day_prices.values
    R = np.diff(np.log(P), axis=0)
    if len(R) < 2:
        continue

    H = 22
    rc = compute_kernel_rcov(R, H)

    indices = np.tril_indices_from(rc)
    rcs.append(rc[indices])
    dates.append(date)


In [18]:
def vec_to_mat(vec, n):
    mat = np.zeros((n, n))
    tril_indices = np.tril_indices(n)
    mat[tril_indices] = vec
    mat = mat + mat.T - np.diag(np.diag(mat))
    return mat

In [20]:
n_total, n_psd, n_nearpsd = 0, 0, 0
for rc_vech in rcs:
    M = vec_to_mat(rc_vech, 8)
    eigs = np.linalg.eigvalsh(M)
    n_total += 1
    if eigs.min() >= 0:
        n_psd += 1
    if eigs.min() >= -1e-6 * eigs.max():
        n_nearpsd += 1
print(f"PSD: {n_psd}/{n_total} = {n_psd/n_total:.1%}")
print(f"Near-PSD (within 1e-6 relative): {n_nearpsd}/{n_total} = {n_nearpsd/n_total:.1%}")

PSD: 3992/3992 = 100.0%
Near-PSD (within 1e-6 relative): 3992/3992 = 100.0%


In [21]:
df_rcs_k = pd.DataFrame(
    rcs,               
    index=pd.to_datetime(dates),
)

output_dir = "data/"
os.makedirs(output_dir, exist_ok=True)
filename = "rc_kernel.txt"
path = os.path.join(output_dir, filename)

#df_rcs_k.to_csv(path, sep='\t', index=False, header=False)

print(f"Saved rcovs to {path}")

Saved rcovs to data/rc_kernel.txt


In [22]:
df_rcs_k

,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,35
2009-01-02,0.000361,0.000307,0.000443,0.000368,0.000368,0.000703,0.000249,0.000251,0.000291,0.000234,...,0.000054,0.000144,0.000245,0.000247,0.000269,0.000217,0.000208,0.000108,0.000126,0.000259
2009-01-05,0.000636,0.000501,0.000737,0.000484,0.000380,0.000536,0.000390,0.000307,0.000390,0.000416,...,0.000114,0.000219,0.000376,0.000323,0.000318,0.000299,0.000267,0.000099,0.000080,0.000340
2009-01-06,0.000531,0.000471,0.000762,0.000425,0.000491,0.000502,0.000341,0.000382,0.000358,0.000324,...,0.000219,0.000391,0.000335,0.000466,0.000383,0.000319,0.000294,0.000188,0.000346,0.000405
2009-01-07,0.000415,0.000368,0.000821,0.000209,0.000275,0.000369,0.000145,0.000243,0.000241,0.000226,...,0.000052,0.000148,0.000120,0.000217,0.000216,0.000191,0.000183,0.000043,0.000047,0.000217
2009-01-08,0.000591,0.000502,0.000624,0.000413,0.000373,0.000482,0.000302,0.000228,0.000268,0.000264,...,0.000041,0.000094,0.000360,0.000274,0.000299,0.000295,0.000304,0.000181,0.000073,0.000412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-20,0.000091,0.000053,0.000069,0.000098,0.000064,0.000136,0.000096,0.000058,0.000118,0.000119,...,0.000048,0.000104,0.000082,0.000043,0.000122,0.000113,0.000160,0.000067,0.000085,0.000210
2024-12-23,0.000047,0.000043,0.000087,0.000032,0.000041,0.000044,0.000035,0.000034,0.000032,0.000036,...,0.000034,0.000059,0.000035,0.000040,0.000025,0.000031,0.000037,0.000029,0.000027,0.000072
2024-12-26,0.000028,0.000020,0.000055,0.000011,0.000014,0.000014,0.000013,0.000015,0.000012,0.000017,...,0.000015,0.000028,0.000002,-0.000002,0.000007,0.000014,0.000022,0.000001,-0.000006,0.000043
2024-12-27,0.000112,0.000135,0.000192,0.000105,0.000125,0.000108,0.000105,0.000127,0.000106,0.000109,...,0.000053,0.000081,0.000070,0.000067,0.000083,0.000083,0.000125,0.000032,0.000046,0.000134


In [23]:
def mat_to_cholesky_vec(mat):
    try:
        L = np.linalg.cholesky(mat)
    except np.linalg.LinAlgError:
        eps = 1e-10
        mat = mat + eps * np.eye(mat.shape[0])
        L = np.linalg.cholesky(mat)
    
    tril_indices = np.tril_indices_from(L)
    return L[tril_indices]

In [24]:
cholesky_rows = []

for _, row in df_rcs.iterrows():
    vec = row.values
    mat = vec_to_mat(vec, 8)
    chol_vec = mat_to_cholesky_vec(mat)
    cholesky_rows.append(chol_vec)

df_chol = pd.DataFrame(
    cholesky_rows,
    index=df_rcs.index
)

In [25]:
df_chol

,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,35
0,0.017417,0.016023,0.012352,0.016271,0.005470,0.015418,0.011586,0.002580,0.002369,0.007971,...,0.002601,0.008421,0.012297,0.002992,0.001784,0.004953,0.002061,0.002019,0.000258,0.007771
1,0.022165,0.019003,0.016741,0.015902,0.003252,0.015275,0.014040,0.002014,0.005499,0.008585,...,0.006391,0.009867,0.014879,0.003585,0.004466,0.004498,0.003541,0.001839,-0.000344,0.007891
2,0.021913,0.018264,0.018680,0.016184,0.006090,0.014804,0.012959,0.003989,0.004849,0.007393,...,0.002591,0.009866,0.012253,0.006682,0.006164,0.003505,0.002776,0.001241,0.002221,0.007681
3,0.021461,0.018325,0.020269,0.013023,0.003167,0.015919,0.009984,0.004167,0.006915,0.008758,...,0.002391,0.008319,0.009694,0.003361,0.005729,0.005181,0.002665,0.000329,0.000254,0.007174
4,0.023088,0.021877,0.014774,0.017835,0.003946,0.013968,0.013135,0.002177,0.005021,0.010945,...,0.002572,0.007438,0.014169,0.002558,0.004240,0.007431,0.004645,0.001310,-0.001276,0.007599
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3987,0.008756,0.006936,0.007192,0.009371,0.001853,0.005183,0.008945,0.001112,0.002502,0.003339,...,0.001446,0.004901,0.009125,0.000558,0.005400,0.003929,0.006416,0.000388,0.000305,0.007328
3988,0.006745,0.006125,0.006543,0.004438,0.000680,0.004559,0.005019,0.000261,0.001965,0.002851,...,0.002056,0.004783,0.004709,-0.000303,0.001866,0.002028,0.003733,0.000609,0.000364,0.005687
3989,0.004770,0.004001,0.005601,0.003006,0.000638,0.003578,0.003095,0.000464,0.002174,0.002274,...,0.001339,0.004138,0.001969,-0.000929,0.004015,0.002327,0.002636,-0.000161,-0.000644,0.004645
3990,0.008243,0.008537,0.005484,0.007828,0.000796,0.003907,0.007711,0.000648,0.002221,0.002328,...,0.000997,0.003899,0.006910,-0.000413,0.004288,0.002128,0.004645,0.000036,0.000268,0.004150


In [26]:
maxi = 0
n = 8

for v in range(len(df_chol)):
    vec = df_chol.iloc[v].values
    L = np.zeros((n, n))
    L[np.tril_indices(n)] = vec
    Sigma_reconstructed = L @ L.T

    matrix = vec_to_mat(df_rcs.iloc[v].values, n)

    m = np.max(np.abs(matrix - Sigma_reconstructed))
    if m > maxi:
        maxi = m
        print(v, maxi)

print("Max:", maxi)

0 5.421010862427522e-20
5 1.0842021724855044e-19
7 2.168404344971009e-19
13 8.673617379884035e-19
Max: 8.673617379884035e-19


In [27]:
cholesky_rows_k = []

for _, row in df_rcs_k.iterrows():
    vec = row.values
    mat = vec_to_mat(vec, 8)
    chol_vec = mat_to_cholesky_vec(mat)
    cholesky_rows_k.append(chol_vec)

df_chol_k = pd.DataFrame(
    cholesky_rows_k,
    index=df_rcs_k.index
)

In [28]:
df_chol_k

,0,1,2,3,4,5,6,7,8,9,...,26,27,28,29,30,31,32,33,34,35
2009-01-02,0.019002,0.016179,0.013457,0.019341,0.004057,0.017684,0.013112,0.002858,0.001459,0.007218,...,0.001723,0.008275,0.012894,0.002831,0.000469,0.005381,0.004101,0.002398,0.001557,0.005487
2009-01-05,0.025228,0.019839,0.018529,0.019204,-0.000031,0.012943,0.015463,-0.000010,0.007154,0.011213,...,0.007166,0.008833,0.014896,0.001483,0.002439,0.004594,0.004396,0.001646,-0.001642,0.008030
2009-01-06,0.023050,0.020414,0.018578,0.018439,0.006173,0.011151,0.014775,0.004312,0.005296,0.007703,...,0.005427,0.009246,0.014519,0.009143,0.005297,0.004743,0.003088,0.002178,0.002422,0.006347
2009-01-07,0.020362,0.018079,0.022226,0.010273,0.004001,0.015734,0.007100,0.005152,0.009342,0.007882,...,0.002594,0.008845,0.005913,0.004939,0.008637,0.005502,0.001978,-0.002160,-0.000752,0.006572
2009-01-08,0.024310,0.020637,0.014059,0.016978,0.001642,0.013824,0.012424,-0.002050,0.004393,0.009294,...,0.001407,0.007349,0.014822,-0.002289,0.003693,0.009632,0.006727,0.001651,-0.001522,0.005492
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-20,0.009514,0.005616,0.006109,0.010279,0.001103,0.005394,0.010115,0.000265,0.002473,0.003281,...,0.000726,0.004695,0.008659,-0.000997,0.006354,0.003080,0.005680,0.001216,-0.000051,0.007111
2024-12-23,0.006880,0.006213,0.006970,0.004614,0.001729,0.004397,0.005065,0.000382,0.001925,0.002530,...,0.002236,0.004474,0.005064,0.001184,-0.000191,0.002179,0.002944,0.000376,0.001106,0.005476
2024-12-26,0.005259,0.003727,0.006443,0.002186,0.000866,0.002850,0.002516,0.000929,0.001863,0.002444,...,0.001210,0.003096,0.000440,-0.000522,0.002338,0.003666,0.001500,-0.000536,-0.001628,0.004286
2024-12-27,0.010571,0.012802,0.005316,0.009922,-0.000349,0.003113,0.009937,-0.000098,0.002339,0.002134,...,0.001753,0.003127,0.006667,-0.003531,0.004864,0.002186,0.005891,-0.000775,-0.000099,0.003608


In [29]:
maxi_k = 0
n = 8

for v in range(len(df_chol_k)):
    vec = df_chol_k.iloc[v].values
    L = np.zeros((n, n))
    L[np.tril_indices(n)] = vec
    Sigma_reconstructed = L @ L.T

    matrix = vec_to_mat(df_rcs_k.iloc[v].values, n)

    m = np.max(np.abs(matrix - Sigma_reconstructed))
    if m > maxi_k:
        maxi_k = m
        print(v, maxi_k)

print("Max:", maxi_k)

0 5.421010862427522e-20
1 1.0842021724855044e-19
7 2.168404344971009e-19
9 8.673617379884035e-19
Max: 8.673617379884035e-19


In [30]:
output_dir = "data/"
os.makedirs(output_dir, exist_ok=True)
filename = "cholesky_kernel.txt"
path = os.path.join(output_dir, filename)

df_chol_k.to_csv(path, sep='\t', index=False, header=False)

print(f"Saved rcovs to {path}")

Saved rcovs to data/cholesky_kernel.txt


In [31]:
output_dir = "data/"
os.makedirs(output_dir, exist_ok=True)
filename = "cholesky_sub.txt"
path = os.path.join(output_dir, filename)

df_chol.to_csv(path, sep='\t', index=False, header=False)

print(f"Saved rcovs to {path}")

Saved rcovs to data/cholesky_sub.txt


In [32]:
df_returns

,XLB,XLE,XLF,XLI,XLK,XLP,XLU,XLY
2009-01-05 16:00:00,-0.004675,0.020566,-0.015180,0.001656,-0.010376,0.001641,0.013038,-0.001985
2009-01-06 16:00:00,0.026497,0.013598,0.019925,0.015901,0.027735,-0.039350,-0.013710,0.019158
2009-01-07 16:00:00,-0.014212,-0.039951,-0.054324,-0.044248,-0.036302,0.012303,-0.008118,-0.040496
2009-01-08 16:00:00,0.004162,0.008185,-0.004183,0.004218,0.013296,-0.017440,-0.004421,0.005883
2009-01-09 16:00:00,-0.022009,-0.032395,-0.037508,-0.021610,-0.022259,-0.012964,-0.008221,-0.031181
...,...,...,...,...,...,...,...,...
2024-12-20 16:00:00,0.012938,0.009791,0.015055,0.013458,0.016248,0.003510,0.014436,0.002629
2024-12-23 16:00:00,-0.001363,0.005855,0.001882,-0.000826,0.009540,-0.005283,0.003794,0.002928
2024-12-26 16:00:00,0.003872,0.007596,0.014356,0.008527,0.010614,0.009709,0.003150,0.019146
2024-12-27 16:00:00,-0.005401,-0.000118,-0.007768,-0.007476,-0.012801,-0.004906,-0.002625,-0.016662
